# 02.01 · Tools: definition, schemas, and docstrings that work

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Module 01 (models, messages, structured output). Lesson 00.02 environment. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what a tool is to a model: a *name*, a *description* and a *JSON Schema* for the arguments. Nothing else is sent.
2. **Create** tools with `@tool`: type hints, docstrings (and `parse_docstring=True`), custom names and descriptions.
3. **Define** rich argument schemas with Pydantic (`Field` descriptions, `Literal` enums, bounds) and see validation happen *before* your code runs.
4. **Know** the reserved argument names (`config`, `runtime`) and what happens if you use them.
5. **Choose** a return value: string, object, or `content_and_artifact` (data for your code that the model never sees).
6. **Write** descriptions that make the model pick the right tool, and **avoid** the `eval` trap in calculator tools.
7. **Jupyter skill:** check which kernel and Python a notebook really uses, and fix `ModuleNotFoundError` for good.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Tools](https://docs.langchain.com/oss/python/langchain/tools) | **Main page.** `@tool`, custom names/descriptions, Pydantic/JSON schemas, reserved names, return values |
| [`@tool` reference](https://reference.langchain.com/python/langchain-core/tools/convert/tool) | Every option: `parse_docstring`, `response_format`, `return_direct`, `args_schema` |
| [Groq: tool use overview](https://console.groq.com/docs/tool-use/overview) | What Groq sends and returns; which models support parallel tool calls (the GPT-OSS models do **not**) |
| [Groq: local tool calling](https://console.groq.com/docs/tool-use/local-tool-calling) | The raw JSON behind every LangChain tool call |
| [Tool integrations](https://docs.langchain.com/oss/python/integrations/tools/index) | Ready-made tools (search, code, databases). Browse it; we build our own first |
| [Installing the IPython kernel](https://ipython.readthedocs.io/en/stable/install/kernel_install.html) · [Jupyter kernels](https://docs.jupyter.org/en/latest/install/kernels.html) | The Jupyter skill of this module |

---
## Step 0 · Install and set up

Nothing new to install: `langchain`, `langchain-core` and `langchain-groq` came with Module 01, and `langgraph` (which `create_agent` runs on) is a dependency of `langchain`. The cell below just makes sure the versions are recent enough.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "langgraph", "pydantic",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Jupyter skill: which kernel is running this notebook?

Module 02 imports the new `mastery.tools` package. If the notebook runs in the wrong environment, you get `ModuleNotFoundError: No module named 'mastery'` (or `'langchain'`), even though "you installed it". Ninety percent of the time the cause is the **kernel**, not the package.

Remember the chain from Lesson 00.02: Jupyter (in `base`) → **kernelspec** `lc-mastery` (a small `kernel.json`) → the Python inside `anaconda3/envs/lc-mastery`. Let's check every link.

In [ ]:
# 🧪 Offline: which Python executes this notebook, and which kernelspec started it?
import json
import os
import sys

print("sys.executable :", sys.executable)
print("sys.prefix     :", sys.prefix)
print("conda env      :", os.environ.get("CONDA_DEFAULT_ENV", "(not reported to the kernel)"))
in_course_env = "lc-mastery" in sys.prefix or ".venv" in sys.prefix
print("course env?    :", "✅ yes" if in_course_env else "❌ no: Kernel → Change Kernel… → Python (lc-mastery)")

try:
    from jupyter_client.kernelspec import KernelSpecManager
    spec = KernelSpecManager().get_kernel_spec("lc-mastery")
    print("kernel.json argv:", spec.argv[0])   # The Python the 'lc-mastery' kernelspec launches.
except Exception as exc:  # Not registered, or jupyter_client missing in this env.
    print("kernelspec 'lc-mastery':", type(exc).__name__, "(register it: Lesson 00.02, Step 6)")

**You should now see** `sys.executable` ending in `envs\\lc-mastery\\python.exe` (Windows) or `envs/lc-mastery/bin/python`, `course env? ✅ yes`, and the kernelspec pointing at the **same** Python.

### Fixing "module not found" for good

| Symptom | Cause | Fix |
|---|---|---|
| `No module named 'mastery'` | Kernel is not `lc-mastery`, or the package was never installed there | Pick **Python (lc-mastery)**; in Git Bash with `(lc-mastery)` active: `python -m pip install -e ".[dev]"`, then **Kernel → Restart** |
| `No module named 'mastery.tools'` but `mastery` works | Installed as a *copy* (no `-e`), so new files in `src/` are invisible | Reinstall with `-e` (editable), restart the kernel |
| Works in Git Bash, fails in the notebook | Two different Pythons | Compare `which python` (Git Bash) with `sys.executable` above |
| The kernelspec points to a deleted/renamed env | Env was recreated | `python -m ipykernel install --user --name lc-mastery --display-name "Python (lc-mastery)"` again |
| `%pip install x` worked but `import x` fails | Package was installed while an older version was already imported | **Kernel → Restart**, then run from the top |

> `%pip` installs into `sys.executable`, the kernel's Python, which is exactly why the course uses it in Step 0. `!pip` runs whatever `pip` comes first on the PATH of `cmd.exe` (Lesson 00.03).

### Why one environment per course?

Every project pins its own versions (`langchain>=1.4.4,<2`, `groq<1`, ...). Put two courses or projects in one environment and an upgrade for one silently breaks the other. One env per course, one kernelspec per env, and a display name that says which (`Python (lc-mastery)`), make "which Python am I in?" a question you can answer at a glance.

Housekeeping commands (Git Bash, any env that has Jupyter):

```bash
jupyter kernelspec list                      # every registered kernel and its folder
jupyter kernelspec remove old-env-name       # delete a stale one (asks for confirmation)
python -m ipykernel install --user --name lc-mastery --display-name "Python (lc-mastery)"   # (re)register: run with lc-mastery ACTIVE
```

The last one must run **inside** the environment you register: `ipykernel install` records the `python` that runs it.

---
## Part 2 · What a tool looks like *to the model*

A model cannot run Python. When you give it tools, the provider receives a **list of JSON descriptions**, and the model may answer with a request: *"please call `get_weather` with `{"city": "Bogotá"}`"*. Your code runs the function and sends the result back (Lesson 02.02 does this by hand).

So a tool has exactly three parts the model sees:

| Part | Comes from | The model uses it to... |
|---|---|---|
| `name` | function name (or `@tool("name")`) | refer to the tool |
| `description` | docstring (or `description=`) | decide **when** to call it |
| `parameters` (JSON Schema) | type hints / `args_schema` | decide **how** to fill the arguments |

The function body, its imports, your comments: none of it is sent. Let's prove it.

In [ ]:
# 🧪 Offline: the simplest tool, and exactly what LangChain sends to Groq for it.
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


@tool
def get_exchange_rate(currency: str) -> float:
    """Return how many Colombian pesos (COP) one unit of `currency` costs today."""
    rates = {"USD": 4000.0, "EUR": 4350.0}     # Illustrative, static numbers.
    return rates[currency.upper()]


print("type       :", type(get_exchange_rate).__name__)   # A StructuredTool (a Runnable!).
print("name       :", get_exchange_rate.name)
print("description:", get_exchange_rate.description)
print("args       :", get_exchange_rate.args)
print()
print(json.dumps(convert_to_openai_tool(get_exchange_rate), indent=2))

### What just happened?

- `@tool` turned the function into a `StructuredTool`, which is also a **Runnable** (Lesson 01.07): it has `invoke`, `batch`, `ainvoke`...
- `convert_to_openai_tool` produced the `{"type": "function", "function": {...}}` block, the format Groq's API takes (it is OpenAI-compatible). `ChatGroq.bind_tools` calls exactly this function.
- The **type hint** `str` became `"type": "string"`. Without type hints, `@tool` cannot build a schema, so the docs say they are **required**.
- The docstring became the description. The comment about static rates did **not** travel: the model has no idea the rates are fake. If it matters, put it in the docstring.

In [ ]:
# 🧪 Offline: a tool is a Runnable. Call it with a dict of arguments, like the model would.
print(get_exchange_rate.invoke({"currency": "usd"}))
print(get_exchange_rate.batch([{"currency": "USD"}, {"currency": "EUR"}]))

---
## Part 3 · Names, descriptions, and argument docs

Three ways to control what the model reads:

1. `@tool("snake_case_name")`: override the name. Prefer `snake_case` with letters, digits, `_` and `-`; some providers reject spaces.
2. `@tool(description="...")`: override the description (the docstring then stays a note for developers).
3. `@tool(parse_docstring=True)`: read a **Google-style** `Args:` section and put each argument's text into the schema. Without it, argument descriptions are lost.

In [ ]:
# 🧪 Offline: compare the schema with and without parse_docstring.
def convert(amount: float, from_currency: str, to_currency: str = "COP") -> float:
    """Convert money between currencies using the course's static rates.

    Args:
        amount: Amount of money in `from_currency`, e.g. 120.5.
        from_currency: ISO 4217 code of the source currency, e.g. "USD".
        to_currency: ISO 4217 code of the target currency. Defaults to "COP".
    """
    return amount


plain = tool(convert)
documented = tool(convert, parse_docstring=True)
renamed = tool("convert_currency", description="Convert an amount between two ISO currency codes.")(convert)

for t in (plain, documented, renamed):
    props = t.tool_call_schema.model_json_schema()["properties"]
    print(f"{t.name:<17} | {t.description[:55]:<55} | amount: {props['amount'].get('description', '(no description)')}")

**You should now see** that only the `parse_docstring=True` version carries *"Amount of money in `from_currency`..."* for the `amount` argument, and that the `description` of `plain` is the **whole** docstring (including the `Args:` block, which wastes tokens on every call).

> **`parse_docstring=True` is strict**: a docstring without a proper `Args:` section raises `ValueError` when the tool is created (`error_on_invalid_docstring=True` by default). That's a feature: you find out at import time, not in production.

---
## Part 4 · Rich schemas with Pydantic: validation before your code runs

Type hints give you types. A Pydantic `args_schema` gives you **descriptions, enums, bounds and defaults**, and LangChain **validates the model's arguments** against it before calling your function. Bad arguments never reach your code.

In [ ]:
# 🧪 Offline: a Pydantic argument schema with an enum and bounds.
from typing import Literal

from pydantic import BaseModel, Field


class ExpenseQuery(BaseModel):
    """Arguments for searching the expense log."""

    category: Literal["groceries", "restaurants", "transport", "housing", "utilities"] = Field(
        description="Spending category to filter on.")
    month: str = Field(description="Month in YYYY-MM format, e.g. 2026-09.", pattern=r"^\d{4}-\d{2}$")
    limit: int = Field(default=5, ge=1, le=20, description="Maximum number of rows to return (1-20).")


@tool(args_schema=ExpenseQuery)
def search_expenses(category: str, month: str, limit: int = 5) -> str:
    """Find expenses in one category for one month, newest first."""
    return f"(pretend) {limit} {category} expenses in {month}"


print(json.dumps(search_expenses.tool_call_schema.model_json_schema()["properties"], indent=2))

In [ ]:
# 🧪 Offline: what happens when the "model" sends bad arguments?
from pydantic import ValidationError

good = {"category": "groceries", "month": "2026-09", "limit": 3}
bad = {"category": "casino", "month": "Sept 2026", "limit": 500}

print(search_expenses.invoke(good))
try:
    search_expenses.invoke(bad)
except ValidationError as exc:
    print(f"\nValidationError with {exc.error_count()} problems:")
    for error in exc.errors():
        print(" -", error["loc"][0], "→", error["msg"])

### What just happened?

- The enum and the `pattern` appear **in the schema**, so the model is told the allowed values up front (fewer mistakes), *and* they are **enforced** if it ignores them.
- The bad call raised `ValidationError` with three problems; the function body never ran. Inside an agent (Lessons 02.03–02.04), this error is sent back to the model as a `ToolMessage`, so it can fix its arguments.
- The `limit` bounds (`ge=1, le=20`) protect *you*: a model can't ask for 500 rows and flood its own context window.

> **JSON Schema dicts work too** (`args_schema={...}`), but you lose Pydantic's validation and get plain dicts. Prefer Pydantic.

---
## Part 5 · Reserved argument names: `config` and `runtime`

Two parameter names are **reserved** by LangChain:

| Name | Reserved for |
|---|---|
| `config` | The `RunnableConfig` LangChain passes internally (callbacks, tags, run IDs) |
| `runtime` | `ToolRuntime`: state, context, store, stream writer (Lesson 02.04) |

The surprising part: defining such a tool does **not** fail. It breaks later, at call time.

In [ ]:
# 🧪 Offline: a reserved name fails at CALL time, not at definition time.
@tool
def save_setting(config: str) -> str:
    """Save a user setting."""
    return f"saved {config!r}"


print("Defined fine. The model would even see an argument:", list(save_setting.args))
try:
    save_setting.invoke({"config": "dark mode"})
except TypeError as exc:
    print("At call time →", type(exc).__name__ + ":", exc)

**You should now see** `TypeError: save_setting() missing 1 required positional argument: 'config'`. LangChain took `config` for itself, so your function never received it. The fix is to rename the argument (`setting`, `options`, ...). Note this rule for code review: *no tool argument named `config` or `runtime`*.

> Older tutorials use `InjectedState`, `InjectedStore`, `InjectedToolCallId` or `get_runtime()`. In LangChain v1 the docs use **one** parameter, `runtime: ToolRuntime`, for all of them ([Tools → State injection](https://docs.langchain.com/oss/python/langchain/tools#state-injection)). Lesson 02.04 uses it.

---
## Part 6 · What a tool returns

The tool's return value becomes a `ToolMessage` that goes back to the model. Your choices:

| Return | The model sees | Use for |
|---|---|---|
| `str` | the text | human-readable results |
| `dict` / list / number | a JSON/str serialization | structured data the model should read field by field |
| `(content, artifact)` with `response_format="content_and_artifact"` | **only** `content` | big or private data for *your code* (rows, images, IDs) |
| `Command(update=...)` | a `ToolMessage` you include | updating agent state (Lesson 02.04) |

Calling a tool with a full **tool call** (`{"name", "args", "id", "type": "tool_call"}`) instead of plain args returns a ready-made `ToolMessage` with the matching `tool_call_id`. That's what agents do.

In [ ]:
# 🧪 Offline: content_and_artifact keeps the bulky part away from the model.
@tool(response_format="content_and_artifact")
def list_big_expenses(min_amount: int) -> tuple[str, list[dict]]:
    """List expenses above a minimum amount (in COP)."""
    rows = [{"merchant": "Arriendo Edificio Nogal", "amount": 1_850_000},
            {"merchant": "Avianca", "amount": 486_200}]
    matches = [r for r in rows if r["amount"] >= min_amount]
    summary = f"{len(matches)} expenses >= {min_amount:,} COP: " + ", ".join(r["merchant"] for r in matches)
    return summary, matches          # (what the model reads, what your code keeps)


tool_call = {"name": "list_big_expenses", "args": {"min_amount": 400_000}, "id": "call_123", "type": "tool_call"}
message = list_big_expenses.invoke(tool_call)
print(type(message).__name__, "| tool_call_id:", message.tool_call_id)
print("content  (sent to the model):", message.content)
print("artifact (stays in Python)  :", message.artifact)

**You should now see** a `ToolMessage` whose `content` is one short sentence while `artifact` holds the full rows. In Project 02 you'll use this to keep raw CSV rows out of the prompt (fewer tokens, less data exposed to the provider) while your notebook still plots them.

---
## Part 7 · Descriptions are prompts: make the model pick the right tool

With several tools, the **description is the only thing** the model uses to choose. Vague descriptions cause wrong picks, invented arguments, or no tool call at all. A good description says:

1. **What** it does, with the domain words users will use ("expenses", "spent", "pesos").
2. **When** to use it, and when *not* to (point to the other tool).
3. **Formats** of inputs and outputs (ISO dates, currency codes, units).
4. **Limits** (what it can't do, max rows).

Below, the same two functions with bad and good descriptions. The 🧪 cell prints what the model reads; the 💸 cell asks a real model to pick.

In [ ]:
# 🧪 Offline: vague vs. specific descriptions for the same two functions.
def _lookup(query: str) -> str:
    return "..."


vague = [
    tool("search", description="Searches stuff.")(_lookup),
    tool("lookup", description="Looks things up.")(_lookup),
]
clear = [
    tool("search_expenses", description=(
        "Search MY personal expense log (purchases, bills, how much I spent, in COP). "
        "Use for questions about my own spending. Not for prices or exchange rates."))(_lookup),
    tool("get_exchange_rate", description=(
        "Get today's static exchange rate between two ISO 4217 currency codes (e.g. USD→COP). "
        "Use only to convert money between currencies; it knows nothing about my expenses."))(_lookup),
]
for label, tools in (("VAGUE", vague), ("CLEAR", clear)):
    print(label)
    for t in tools:
        print(f"  {t.name:<18} {t.description}")

In [ ]:
# 💸 Live: which tool does the model pick for each question? (2 tools x 3 questions = 6 short calls)
QUESTIONS = ["How much did I spend on groceries in September?",
             "How many pesos is 25 dollars?",
             "What was my biggest bill last month?"]

if HAVE_GROQ_KEY:
    from mastery.models import get_model

    model = get_model("fast", temperature=0, reasoning_effort="low")
    for label, tools in (("VAGUE", vague), ("CLEAR", clear)):
        bound = model.bind_tools(tools)            # Lesson 02.02 explains bind_tools.
        picks = []
        for question in QUESTIONS:
            reply = bound.invoke(question)
            picks.append(reply.tool_calls[0]["name"] if reply.tool_calls else "(no tool)")
        print(f"{label}: {picks}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** the CLEAR set pick `search_expenses`, `get_exchange_rate`, `search_expenses`. The VAGUE set often picks inconsistently or skips tools. Results vary from run to run, which is itself the lesson: with vague descriptions, behavior is a coin flip.

---
## Part 8 · The course's tool kit: `mastery.tools`

Module 02 adds tested, reusable tools to the course package. Read them: they are short, and each one encodes a lesson.

| Tool | File | The lesson inside |
|---|---|---|
| `calculator` | [`src/mastery/tools/calculator.py`](../../src/mastery/tools/calculator.py) | Exact math via a safe `ast` evaluator, **not `eval`** |
| `get_today`, `days_between`, `add_days` | [`src/mastery/tools/dates.py`](../../src/mastery/tools/dates.py) | Models don't know today's date; ISO dates only |
| `make_csv_lookup_tool(table)` | [`src/mastery/tools/csv_lookup.py`](../../src/mastery/tools/csv_lookup.py) | The developer picks the file; the model picks a column (an enum) and a value |

### Why not `eval`?

The official Tools page shows a one-line calculator: `return str(eval(expression))`. It's a fine illustration of `description=`, but **never ship it**. `eval` runs any Python. The "expression" comes from the model, and the model's input can come from a web page, an email or a PDF someone else wrote (prompt injection, Lesson 04.05). One string such as `__import__('os').remove('important.xlsx')` and the tool deletes a file with **your** permissions.

In [ ]:
# 🧪 Offline: the safe calculator accepts arithmetic and rejects everything else.
from mastery.tools import calculator, safe_eval

for expression in ["(1850000 + 119900) * 0.19", "2 ^ 10", "1 / 0", "__import__('os').getcwd()"]:
    message = calculator.invoke({"name": "calculator", "args": {"expression": expression}, "id": "c", "type": "tool_call"})
    print(f"{expression:<30} → {message.status:<7} {message.content}")

**You should now see** two numbers (`374281.0`, `1024`) and two `error` results, *"Division by zero."* and *"Only numbers and + - * / // % ** ( ) are allowed; found Call."*. The tools set `handle_tool_error = True`, so a `CalculatorError` (a `ToolException`) comes back as an error **message** the model can read and fix, instead of crashing the program.

In [ ]:
# 🧪 Offline: date tools and a CSV lookup tool built for ONE file.
from mastery.tools import CsvTable, days_between, get_today, make_csv_lookup_tool, set_today

set_today("2026-10-12")            # Freeze "today" so this output is reproducible.
print(get_today.invoke({}), "|", days_between.invoke({"start": "2026-10-12", "end": "2026-12-25"}), "days to Christmas")
set_today(None)                    # Back to the real clock.

expenses = CsvTable.from_path(REPO_ROOT / "data" / "samples" / "expenses" / "expenses-2026.csv")
find_expenses = make_csv_lookup_tool(expenses, name="find_expenses", max_results=3)
print(find_expenses.name, "→ column enum:", find_expenses.tool_call_schema.model_json_schema()["properties"]["column"].get("enum"))
result = find_expenses.invoke({"name": "find_expenses", "args": {"column": "merchant", "value": "uber"},
                               "id": "c2", "type": "tool_call"})
print(result.content[:240], "...")
print("rows kept as artifact:", len(result.artifact))

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `ValueError: Function must have a docstring if description not provided` | No docstring and no `description=` | Write the docstring: it *is* the tool's prompt |
| Model never calls your tool | Vague name/description, or the question doesn't match its words | Rewrite the description (Part 7); test with 5–10 real questions |
| `TypeError: missing 1 required positional argument: 'config'` | Reserved argument name | Rename it; use `runtime: ToolRuntime` for injected data |
| Argument descriptions missing in the schema | Docstring `Args:` without `parse_docstring=True` | Add `parse_docstring=True`, or use `Field(description=...)` |
| `ValidationError` inside an agent | Model sent wrong types/values | Expected: the agent returns it to the model. Make the schema clearer (enums, examples) |
| Huge tool results, high token bills | Returning whole tables to the model | Summaries in `content`, rows in `artifact`; cap results |
| Space or dot in the tool name rejected by the provider | Names must be simple identifiers | `snake_case` only |

## ⚠️ Production notes

> ⚠️ **Production note: a tool runs with your permissions.**
> Anything the model can make a tool do, an attacker who controls the model's *input* can make it do. In production: give each tool the narrowest capability (read-only, one table, one folder), validate arguments with schemas and allow-lists, never `eval`/`exec`/`shell=True`, run risky tools in a sandbox (Lesson 11.09), and log every call with its arguments.

> ⚠️ **Production note: tool descriptions are part of your prompt.**
> Changing a docstring changes agent behavior as much as changing the system prompt. Version tool definitions with the code, review description changes like prompt changes, and re-run your agent evaluation set (Module 10) after edits.

## 🏋️ Exercises

1. **(Easy)** Add a `currency` argument to `get_exchange_rate` typed as `Literal["USD", "EUR", "MXN"]`. Print `convert_to_openai_tool(...)` and find the enum.
2. **(Easy)** Call `calculator` with `"17.5% of 2340"`. Read the error. How would you change the *description* so the model sends `2340 * 0.175` instead?
3. **(Medium)** Write `@tool(parse_docstring=True) def split_bill(total: float, people: int, tip_percent: float = 10) -> str` with a Pydantic-free signature, then rewrite it with an `args_schema` that enforces `people >= 1` and `0 <= tip_percent <= 30`. Show the `ValidationError` for `people=0`.
4. **(Stretch)** Use `make_csv_lookup_tool` on `data/samples/support-tickets.jsonl`... it's JSONL, not CSV! Write a tiny `JsonlTable` with the same `where()` method and reuse the same idea (developer picks the file, model picks the field).

<details>
<summary><b>Show solutions</b></summary>

**2.** The error says only numbers and operators are allowed. Add to the description: *"Write percentages as multiplication: 17.5% of 2340 → 2340 * 0.175."* Examples in descriptions are one of the most effective fixes.

**3.**
```python
class SplitArgs(BaseModel):
    total: float = Field(gt=0, description="Bill total in COP.")
    people: int = Field(ge=1, description="Number of people, at least 1.")
    tip_percent: float = Field(default=10, ge=0, le=30, description="Tip in percent, 0-30.")

@tool(args_schema=SplitArgs)
def split_bill(total: float, people: int, tip_percent: float = 10) -> str:
    """Split a restaurant bill evenly, tip included."""
    return f"{total * (1 + tip_percent / 100) / people:,.0f} COP each"

try:
    split_bill.invoke({"total": 120000, "people": 0})
except ValidationError as exc:
    print(exc.errors()[0]["msg"])    # Input should be greater than or equal to 1
```

**4.** Read lines with `json.loads`, collect keys as `columns`, and copy `where()` from `CsvTable`. The tool factory works unchanged if your class has `name`, `columns` and `where`.
</details>

## ✅ Checkpoint

- [ ] I can name the three things a model sees about a tool, and print them with `convert_to_openai_tool`.
- [ ] I can write a tool with `@tool`, `parse_docstring=True`, and a Pydantic `args_schema`, and I know validation happens before my code runs.
- [ ] I know `config` and `runtime` are reserved, and what breaks if I use them.
- [ ] I can return `content_and_artifact` and explain who sees what.
- [ ] I can explain why a calculator tool must not use `eval`.
- [ ] My notebook runs on **Python (lc-mastery)**, and I can fix `ModuleNotFoundError` by checking the kernel first.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open the [`@tool` reference](https://reference.langchain.com/python/langchain-core/tools/convert/tool) and find **one** option this lesson did not use (for example `extras`, `infer_schema` or `error_on_invalid_docstring`). What does it do, and when would you need it? Then check Groq's [tool use overview](https://console.groq.com/docs/tool-use/overview): which current models support **parallel** tool calls? Log both in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A tool, to the model, is a name, a description and a JSON Schema. Everything else stays on your machine.
- `@tool` + type hints + a good docstring is enough to start; `parse_docstring=True` and Pydantic schemas add argument docs, enums, bounds and validation.
- `config` and `runtime` are reserved names; the failure shows up at call time.
- Return text for the model, objects for structured reading, and artifacts for data only your code should see.
- Descriptions are prompts. Tools run with your permissions: narrow, validated, never `eval`.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 02.02 · Tool calling without an agent** → [`02-02-tool-calling-loop.ipynb`](02-02-tool-calling-loop.ipynb).

[Module 02 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)